# Comparing Token Costs for PDF Q&A with Claude
*Understanding the cost-efficiency of Docling preprocessing vs. direct PDF upload*

## Lab 5: Token Cost Comparison

Welcome to Lab 5 in our Docling workshop series! In this lab, you'll explore the economics of using Claude for PDF question-answering by comparing two approaches:

1. **Direct PDF Upload**: Upload a PDF directly to Claude (desktop app or website)
2. **Docling Preprocessing**: Extract text with Docling first, then use the extracted text with Claude

### Why This Matters

When you upload a PDF directly to Claude, each page is processed as both an image and text, consuming **1,500-3,000 tokens per page**. For a 50-page document, that's 75,000-150,000 tokens just for the document context!

By preprocessing with Docling, you can potentially reduce token usage significantly while maintaining high-quality responses for text-heavy documents.

## Learning Objectives

By the end of this lab, you will:
- **Understand** how Claude processes PDFs internally (vision-based conversion)
- **Calculate** token costs for both direct upload and text extraction approaches
- **Compare** estimated costs using current Claude model pricing
- **Evaluate** trade-offs between cost, convenience, and response quality
- **Apply** this knowledge to choose the right approach for your use cases

---

## Prerequisites

Before we begin, ensure you have:
- Python 3.10 or later installed
- Completed Labs 1-2 (or equivalent Docling knowledge)
- A PDF document you'd like to analyze
- Access to Claude (desktop app or claude.ai)
- (Optional) An Anthropic API key for accurate token counting

## Installation and Setup

### Setting up the environment

Ensure you are running Python 3.10 or later in a freshly created virtual environment.

In [ ]:
import sys
assert sys.version_info >= (3, 10) and sys.version_info < (3, 14), "Use Python 3.10, 3.11, 3.12, or 3.13 to run this notebook."

### Install Dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install docling anthropic PyPDF2 httpx
! echo "::endgroup::"

### Import Essential Components

In [ ]:
import os
import tempfile
from pathlib import Path
from typing import Optional

# Docling imports
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

# Anthropic SDK for token counting
import anthropic

# For PDF page counting
from PyPDF2 import PdfReader

# For downloading PDFs from URLs
import httpx

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

print("Setup complete!")

---

## Understanding How Claude Processes PDFs

### How Direct PDF Upload Works

When you upload a PDF directly to Claude (via the desktop app, claude.ai, or API), here's what happens:

1. **Page-to-Image Conversion**: Each page of the PDF is rendered as an image
2. **Text Extraction**: Text is also extracted from each page
3. **Combined Processing**: Claude analyzes both the images and extracted text together

### Token Cost Implications

According to [Anthropic's documentation](https://docs.anthropic.com/en/docs/build-with-claude/pdf-support):
- **Per page**: Approximately **1,500-3,000 tokens** (depending on content density)
- **This includes**: Both the image representation and extracted text
- **Visual elements**: Charts, diagrams, and images are preserved and analyzable

### The Alternative: Text-Only Processing

With Docling preprocessing:
- **Text extraction**: Only the text content is extracted (with structure preserved)
- **Token count**: Based purely on the actual text content
- **Trade-off**: Visual elements may lose some context, but tables and structure are well-preserved

---

## Step 1: Provide Your PDF

Enter the path to your PDF file below. This can be:
- A local file path (e.g., `/path/to/document.pdf`)
- A URL to a PDF (e.g., `https://arxiv.org/pdf/2501.17887`)

### Your turn: use a document that matches your real workload

Token costs scale with document size, and the Docling-vs-direct-upload tradeoff depends heavily on page count and content type. The realistic way to decide is to run this analysis on a document that looks like what you'll actually send to Claude.

<details>
<summary>What should I try?

Good alternatives:
- **A short contract** (5-10 pages) — direct upload often wins here because overhead dominates.
- **A long annual report** (50+ pages) — Docling preprocessing usually pulls ahead.
- **An image-heavy slide deck** — direct upload's vision tokens balloon; good for seeing the break-even.
- **A local PDF** — uncomment Option 1 and use a file path.

The rest of the notebook will re-run its cost math against whatever you pick.

</details>


In [ ]:
# ============================================
# CONFIGURE YOUR PDF HERE
# ============================================

# Option 1: Local file path
# pdf_source = "/path/to/your/document.pdf"

# TODO: try swapping pdf_source to a document representative of your real workload.
# Option 2: URL (default example - Docling technical report)
pdf_source = "https://arxiv.org/pdf/2501.17887"  # <-- change me

# ============================================
print(f"PDF source: {pdf_source}")


---

## Step 2: Estimate Tokens for Direct PDF Upload

When you upload a PDF directly to Claude, each page is processed as both an image and text. Let's estimate the token cost based on page count.

In [ ]:
def get_pdf_page_count(pdf_source: str) -> int:
    """Get the number of pages in a PDF (supports both local files and URLs)."""
    if pdf_source.startswith(('http://', 'https://')):
        # Download the PDF temporarily
        print(f"Downloading PDF from URL...")
        response = httpx.get(pdf_source, follow_redirects=True, timeout=60)
        response.raise_for_status()
        
        with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp:
            tmp.write(response.content)
            tmp_path = tmp.name
        
        try:
            reader = PdfReader(tmp_path)
            page_count = len(reader.pages)
        finally:
            os.unlink(tmp_path)  # Clean up
        return page_count
    else:
        reader = PdfReader(pdf_source)
        return len(reader.pages)


def estimate_direct_upload_tokens(page_count: int, 
                                   tokens_per_page_low: int = 1500,
                                   tokens_per_page_high: int = 3000) -> dict:
    """
    Estimate token count for direct PDF upload to Claude.
    
    Based on Anthropic documentation: Each page typically uses 1,500-3,000 tokens
    depending on content density.
    """
    return {
        "page_count": page_count,
        "tokens_low_estimate": page_count * tokens_per_page_low,
        "tokens_high_estimate": page_count * tokens_per_page_high,
        "tokens_average_estimate": page_count * ((tokens_per_page_low + tokens_per_page_high) // 2)
    }

In [ ]:
# Get page count and estimate direct upload tokens
page_count = get_pdf_page_count(pdf_source)
direct_upload_estimate = estimate_direct_upload_tokens(page_count)

print(f"PDF Analysis:")
print(f"  Total pages: {direct_upload_estimate['page_count']}")
print(f"\nDirect Upload Token Estimates:")
print(f"  Low estimate (sparse content):  {direct_upload_estimate['tokens_low_estimate']:,} tokens")
print(f"  High estimate (dense content):  {direct_upload_estimate['tokens_high_estimate']:,} tokens")
print(f"  Average estimate:               {direct_upload_estimate['tokens_average_estimate']:,} tokens")

---

## Step 3: Extract Text with Docling

Now let's use Docling to extract the text content from the PDF. Docling preserves document structure, tables, and formatting while extracting clean text.

In [ ]:
# Configure Docling converter
pipeline_options = PdfPipelineOptions(
    do_ocr=False,  # Set to True if dealing with scanned documents
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# Convert the PDF
print("Converting PDF with Docling...")
result = converter.convert(pdf_source)
doc = result.document

# Export to markdown (preserves structure)
markdown_text = doc.export_to_markdown()

# Save the extracted text
output_file = output_dir / "extracted_text.md"
doc.save_as_markdown(output_file)

print(f"\nExtraction complete!")
print(f"  Document title: {doc.name}")
print(f"  Pages processed: {len(doc.pages)}")
print(f"  Tables found: {len(doc.tables)}")
print(f"  Pictures found: {len(doc.pictures)}")
print(f"  Character count: {len(markdown_text):,}")
print(f"\nExtracted text saved to: {output_file}")

---

## Step 4: Count Tokens Using Anthropic SDK

Now let's count the actual tokens in the extracted text using Anthropic's official token counting API.

> **Note**: If you have an `ANTHROPIC_API_KEY` environment variable set, we'll use the official API for accurate counting. Otherwise, we'll provide an estimate based on character count.

In [ ]:
def count_tokens_with_anthropic(text: str, model: str = "claude-sonnet-4-5-20250929") -> dict:
    """
    Count tokens using Anthropic's official token counting API.
    
    If no API key is available, provides an estimate based on character count.
    """
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    
    if not api_key:
        # Fallback: estimate based on typical token-to-character ratio
        # Claude uses roughly 4 characters per token on average for English text
        estimated_tokens = len(text) // 4
        return {
            "token_count": estimated_tokens,
            "method": "estimated (no API key)",
            "note": "Set ANTHROPIC_API_KEY environment variable for accurate counting"
        }
    
    try:
        client = anthropic.Anthropic(api_key=api_key)
        
        # Use the count_tokens endpoint
        response = client.messages.count_tokens(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": text
                }
            ]
        )
        
        return {
            "token_count": response.input_tokens,
            "method": "anthropic_api",
            "model": model
        }
    except Exception as e:
        # Fallback to estimation on error
        estimated_tokens = len(text) // 4
        return {
            "token_count": estimated_tokens,
            "method": "estimated (API error)",
            "error": str(e)
        }

In [ ]:
# Count tokens in the extracted text
docling_token_result = count_tokens_with_anthropic(markdown_text)

print(f"Docling Extracted Text Token Count:")
print(f"  Token count: {docling_token_result['token_count']:,}")
print(f"  Counting method: {docling_token_result['method']}")
if 'note' in docling_token_result:
    print(f"  Note: {docling_token_result['note']}")
if 'error' in docling_token_result:
    print(f"  Error: {docling_token_result['error']}")

---

## Step 5: Compare Costs

Let's compare the token costs and estimated pricing for both approaches using current Claude model pricing.

### Your turn: compare models you actually use

The cost comparison below is only as useful as the models in `PRICING`. Edit the dict to reflect the model (or models) you plan to use in production — or add a non-Anthropic model at its published rates to see where the cost lines cross.

<details>
<summary>What should I try?

Useful experiments:
- **Add Claude Opus** (`"claude-opus-4-5": {"input": 15.00, "output": 75.00}`) to see how a premium model shifts the break-even point between direct upload and Docling preprocessing.
- **Adjust to batch-API pricing** (50% off) to see whether Docling still pays off when you're batching.
- **Model you're evaluating** — paste in whichever model you're benchmarking against this workload.

Always verify current pricing at https://www.anthropic.com/pricing — the numbers here rot quickly.

</details>


In [ ]:
# Current Claude pricing (per million tokens)
# IMPORTANT: Verify current pricing at https://www.anthropic.com/pricing
# Prices below are estimates and may be outdated.
# TODO: try adding a model to PRICING (or editing the rates) to compare cost against your workload.
PRICING = {
    "claude-sonnet-4-5": {"input": 3.00, "output": 15.00},  # <-- change me
    "claude-haiku-4-5": {"input": 1.00, "output": 5.00},
}

def calculate_cost(tokens: int, model: str = "claude-sonnet-4-5", is_input: bool = True) -> float:
    """Calculate cost in USD for a given number of tokens."""
    price_per_million = PRICING[model]["input" if is_input else "output"]
    return (tokens / 1_000_000) * price_per_million

def format_cost(cost: float) -> str:
    """Format cost with appropriate precision."""
    if cost < 0.01:
        return f"${cost:.4f}"
    return f"${cost:.3f}"


In [ ]:
# Calculate costs for comparison
docling_tokens = docling_token_result['token_count']
direct_low = direct_upload_estimate['tokens_low_estimate']
direct_high = direct_upload_estimate['tokens_high_estimate']
direct_avg = direct_upload_estimate['tokens_average_estimate']

print("=" * 70)
print("TOKEN COST COMPARISON")
print("=" * 70)

print(f"\n{'Approach':<35} {'Tokens':>15} {'% of Direct Avg':>15}")
print("-" * 70)
print(f"{'Direct Upload (low estimate)':<35} {direct_low:>15,}")
print(f"{'Direct Upload (average estimate)':<35} {direct_avg:>15,} {'100%':>15}")
print(f"{'Direct Upload (high estimate)':<35} {direct_high:>15,}")
print(f"{'Docling Text Extraction':<35} {docling_tokens:>15,} {docling_tokens/direct_avg*100:>14.1f}%")

# Token savings
savings = direct_avg - docling_tokens
savings_pct = (savings / direct_avg) * 100 if direct_avg > 0 else 0

print(f"\n{'Token Savings with Docling:':<35} {savings:>15,} ({savings_pct:.1f}%)")

In [ ]:
print("\n" + "=" * 70)
print("ESTIMATED COSTS BY MODEL (per query with this document)")
print("=" * 70)

for model_name, prices in PRICING.items():
    print(f"\n{model_name.upper()}:")
    direct_cost = calculate_cost(direct_avg, model_name)
    docling_cost = calculate_cost(docling_tokens, model_name)
    print(f"  {'Direct Upload (avg):':<30} {format_cost(direct_cost):>10}")
    print(f"  {'Docling Extraction:':<30} {format_cost(docling_cost):>10}")
    cost_savings = direct_cost - docling_cost
    print(f"  {'Savings per query:':<30} {format_cost(cost_savings):>10}")

---

## Step 6: Try Both Approaches with Claude

Now let's put this into practice! We'll guide you through testing both approaches with the same questions so you can compare response quality.

### Sample Questions to Try

Use these questions to test both approaches and compare the results:

In [ ]:
# Sample questions for testing both approaches
sample_questions = [
    "What is the main topic or purpose of this document?",
    "Summarize the key findings or conclusions in 3-5 bullet points.",
    "What are the most important figures, tables, or statistics mentioned?",
    "Who are the authors or contributors to this document?",
    "What methodology or approach is described in this document?",
]

print("Sample Questions to Ask Claude:")
print("-" * 50)
for i, q in enumerate(sample_questions, 1):
    print(f"{i}. {q}")

print("\n" + "-" * 50)
print("Try asking the same question using both approaches below!")

### Approach A: Direct PDF Upload

**Instructions:**

1. Open Claude (desktop app or [claude.ai](https://claude.ai))
2. Click the attachment/upload button (📎)
3. Select your PDF file
4. Ask one of the sample questions above
5. Note the response quality

**What to observe:**
- Can Claude see and interpret charts/graphs?
- Does Claude reference specific visual elements?
- How detailed is the response?
- Does Claude mention page numbers or specific locations?

### Approach B: Docling Extracted Text

**Instructions:**

1. Run the cell below to display the extracted text
2. Copy the text (or open the saved file)
3. Open a **new** Claude conversation
4. Paste the text and ask the **same question** as Approach A
5. Compare the response with Approach A

**What to observe:**
- Is the response equally detailed?
- Are there any gaps due to missing visual elements?
- Does the structure (headers, tables) come through clearly?
- How does response quality compare overall?

In [ ]:
# Display a preview of the extracted text
preview_length = 3000
print("Extracted Text Preview:")
print("=" * 70)
print(markdown_text[:preview_length])
if len(markdown_text) > preview_length:
    print(f"\n... [{len(markdown_text) - preview_length:,} more characters]")
print("=" * 70)

print(f"\nFull extracted text saved to: {output_file}")
print("You can open this file and copy the contents to paste into Claude.")

---

## Step 7: Evaluate Response Quality

After trying both approaches, use this framework to compare your results.

In [ ]:
# Quality evaluation template
evaluation_template = """
RESPONSE QUALITY COMPARISON
============================

Question Asked: [Your question here]

DIRECT PDF UPLOAD:
------------------
Response Quality (1-5): [ ]
Visual Elements Referenced: [Yes/No]
Key Information Captured: [Yes/No/Partial]
Notes: 

DOCLING EXTRACTED TEXT:
-----------------------
Response Quality (1-5): [ ]
Visual Elements Referenced: [N/A - text only]
Key Information Captured: [Yes/No/Partial]
Notes: 

OVERALL ASSESSMENT:
-------------------
Which approach provided better answers? [ Direct / Docling / Similar ]
Was the cost difference worth it? [ Yes / No / Depends ]
Recommended approach for this document type: [ ]
"""

print(evaluation_template)

---

## Summary and Recommendations

### When to Use Direct PDF Upload

**Best for:**
- Documents with important visual elements (charts, diagrams, infographics)
- When layout and visual formatting carry meaning
- Quick one-off analyses where setup time matters more than cost
- Documents with complex tables that benefit from visual rendering
- Scanned documents or image-heavy PDFs

**Trade-offs:**
- Higher token cost (1,500-3,000 tokens per page)
- No preprocessing required
- Full visual context preserved

### When to Use Docling Preprocessing

**Best for:**
- Text-heavy documents (reports, articles, contracts, research papers)
- Repeated queries on the same document (extract once, query many times)
- Cost-sensitive applications with high document volumes
- Integration into automated pipelines
- When you need to process the text further (chunking, embeddings, etc.)

**Trade-offs:**
- Lower token cost (based on actual text content)
- Requires preprocessing step
- Visual elements may lose context (though Docling preserves structure well)

### Cost Optimization Strategies

1. **Prompt Caching**: If using the API with the same document repeatedly, enable prompt caching for up to 90% discount on cached tokens
2. **Batch Processing**: Use the Batch API for 50% discount on high-volume processing
3. **Model Selection**: Use Haiku for simple extraction, Sonnet for analysis, Opus for complex reasoning
4. **Hybrid Approach**: Extract text with Docling, but include key images separately when visual context is critical

In [ ]:
# Print final summary
print("=" * 70)
print("FINAL SUMMARY FOR YOUR DOCUMENT")
print("=" * 70)

print(f"\nDocument: {doc.name}")
print(f"Pages: {page_count}")

print(f"\nToken Comparison:")
print(f"  Direct Upload (estimated): {direct_avg:,} tokens")
print(f"  Docling Extraction:        {docling_tokens:,} tokens")
print(f"  Potential Savings:         {savings:,} tokens ({savings_pct:.1f}%)")

print(f"\nCost Comparison (Claude Sonnet 4.5):")
direct_cost = calculate_cost(direct_avg, "claude-sonnet-4-5")
docling_cost = calculate_cost(docling_tokens, "claude-sonnet-4-5")
print(f"  Direct Upload:      {format_cost(direct_cost)} per query")
print(f"  Docling Extraction: {format_cost(docling_cost)} per query")

print("\n" + "-" * 70)
if savings_pct > 50:
    print(f"Recommendation: Docling preprocessing offers significant savings ({savings_pct:.0f}%)")
    print("Consider using this approach for text-heavy documents.")
elif savings_pct > 20:
    print(f"Recommendation: Moderate savings with Docling ({savings_pct:.0f}%)")
    print("Choose based on whether visual context is important for your use case.")
else:
    print(f"Recommendation: Minimal token savings ({savings_pct:.0f}%)")
    print("Direct upload may be more convenient for this document type.")

---

## Resources

### Documentation
- [Anthropic PDF Support](https://docs.anthropic.com/en/docs/build-with-claude/pdf-support)
- [Anthropic Token Counting API](https://docs.anthropic.com/en/docs/build-with-claude/token-counting)
- [Claude Pricing](https://www.anthropic.com/pricing)
- [Docling Documentation](https://docling-project.github.io/docling/)

### Related Labs
- **Lab 1**: Document Conversion with Docling
- **Lab 2**: Intelligent Chunking Strategies
- **Lab 3**: Multimodal RAG with Visual Grounding
- **Lab 4**: Docling as a Service